# Build Central Raw Dataset

Notebook นี้ใช้สร้าง `data/raw/central/central_raw_dataset.csv` จากไฟล์ CSV ใน `data/raw/pre-combine` โดยยังไม่ clean หรือแปลงหน่วย เพื่อไม่ให้ row จากคนละ source ผิดความหมาย

หลักการรวมข้อมูลรอบนี้:
- append ทุก row จากทุกไฟล์ที่อยู่ใน `data/raw/pre-combine`
- map ชื่อคอลัมน์หลักเข้า schema กลาง เช่น `label`, `area`, `soil_type`, `ph`, `n`, `p`, `k`, `humidity`, `soil_moisture`, `temperature`, `rainfall`
- เก็บ unit แยกต่อ feature เช่น `n_unit`, `p_unit`, `k_unit`, `rainfall_unit`
- ถ้า source ไม่มีค่านั้น ให้เว้นว่างไว้
- เก็บคอลัมน์ต้นฉบับทุกตัวไว้ใน `raw__...` เพื่อ audit ย้อนกลับได้


In [ ]:
from pathlib import Path
import runpy
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd()
INPUT_DIR = PROJECT_ROOT / "data" / "raw" / "pre-combine"
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "central"

print(PROJECT_ROOT)
print(INPUT_DIR)


## 1. Source files

ตรวจไฟล์ต้นทางที่ script จะนำมารวม ถ้าเพิ่มไฟล์ CSV ใหม่ใน `pre-combine` แล้ว rerun notebook นี้ script จะดึงเข้ามาด้วยโดยอัตโนมัติ


In [ ]:
source_files = sorted(INPUT_DIR.glob("*.csv"))
source_overview = []
for path in source_files:
    df = pd.read_csv(path)
    source_overview.append({
        "source_file": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 4),
        "path": path.relative_to(PROJECT_ROOT).as_posix(),
    })

source_overview = pd.DataFrame(source_overview)
display(source_overview)
print("Total source rows:", int(source_overview["rows"].sum()))


## 2. Build central raw CSV

รัน script สร้างไฟล์กลาง โดย script จะไม่แปลงหน่วย N/P/K หรือ rainfall แต่จะใส่ unit metadata ไว้ข้างๆ ค่าแทน


In [ ]:
runpy.run_path(str(PROJECT_ROOT / "scripts" / "build_central_raw_dataset.py"), run_name="__main__")


## 3. Validate output shape

เช็กว่า row count ในไฟล์กลางเท่ากับผลรวม row จาก source files และ `central_raw_id` ไม่ซ้ำ


In [ ]:
central_path = OUTPUT_DIR / "central_raw_dataset.csv"
summary_path = OUTPUT_DIR / "central_raw_dataset_summary.csv"
unit_path = OUTPUT_DIR / "central_unit_dictionary.csv"

central = pd.read_csv(central_path, low_memory=False)
summary = pd.read_csv(summary_path)
unit_dictionary = pd.read_csv(unit_path)

print("Central rows:", len(central))
print("Source rows:", int(source_overview["rows"].sum()))
print("Central columns:", len(central.columns))
print("central_raw_id unique:", central["central_raw_id"].is_unique)
print("source_file + source_row_id duplicates:", int(central.duplicated(["source_file", "source_row_id"]).sum()))

assert len(central) == int(source_overview["rows"].sum())
assert central["central_raw_id"].is_unique


## 4. Summary by source

ตารางนี้ช่วยดูว่าแต่ละไฟล์ถูกดึงเข้ามากี่ row และ feature หลักมีค่าไม่ว่างกี่ row พร้อม unit ของ N/P/K และ rainfall


In [ ]:
summary_cols = [
    "source_file", "rows", "label_non_null", "area_non_null", "soil_type_non_null",
    "ph_non_null", "n_non_null", "p_non_null", "k_non_null",
    "humidity_non_null", "soil_moisture_non_null", "temperature_non_null", "rainfall_non_null",
    "n_unit", "p_unit", "k_unit", "rainfall_unit",
]
display(summary[summary_cols])


## 5. Unit dictionary

ดู mapping หน่วยของแต่ละ source column โดยเฉพาะคอลัมน์ที่ห้ามเอาไปบวก/ต่อความหมายกันตรงๆ เช่น N/P/K และ rainfall


In [ ]:
focus_columns = ["n", "p", "k", "rainfall", "area", "soil_moisture", "temperature", "humidity"]
display(
    unit_dictionary[unit_dictionary["source_column"].isin(focus_columns)]
    .sort_values(["source_column", "source_file"])
    .reset_index(drop=True)
)


## 6. Preview central raw data

คอลัมน์แรกๆ คือ schema กลาง ส่วนคอลัมน์ `raw__...` คือค่าต้นฉบับจากไฟล์เดิม เก็บไว้เพื่อเทียบย้อนกลับก่อนตัดสินใจ clean/convert ในรอบถัดไป


In [ ]:
canonical_preview_cols = [
    "central_raw_id", "source_file", "source_row_id", "label", "area", "soil_type",
    "soil_color", "ph", "n", "n_unit", "p", "p_unit", "k", "k_unit",
    "humidity", "soil_moisture", "temperature", "rainfall", "rainfall_unit",
]
display(central[canonical_preview_cols].head(20))
